<a href="https://colab.research.google.com/github/WellingtonRoque/Algoritmos/blob/main/Aula9_SQLite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 📘 Aula: SQLite com Python – do Básico ao CRUD

## 🎯 Objetivos
- Compreender o que é o SQLite e suas vantagens.
- Conhecer os principais comandos SQL.
- Aprender a usar SQLite com Python.
- Desenvolver uma aplicação CRUD (Create, Read, Update, Delete).



## 1. O que é SQLite?
- Banco de dados **relacional** (dados em tabelas).
- **Leve**: não precisa de servidor, os dados ficam em um arquivo `.db`.
- Usado em **aplicativos mobile**, **softwares desktop** e **prototipagem**.
- Biblioteca `sqlite3` já vem com o Python.


## 2. Tipos de Dados no SQLite

O SQLite utiliza *storage classes* para representar tipos comuns.

| Tipo | Descrição |
|------|-----------|
| NULL | Valor nulo |
| INTEGER | Número inteiro |
| REAL | Número decimal |
| TEXT | Texto |
| BLOB | Dados binários |

# 📌 Criando célula de código – Criar tabelas




In [ ]:
# Conectando ao SQLite e criando o banco
import sqlite3

# Cria ou conecta ao banco
con = sqlite3.connect("meubanco.db")
cur = con.cursor()

# Cria a tabela usuarios
cur.execute("""
CREATE TABLE IF NOT EXISTS usuarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL
);
""")

# Cria a tabela tarefas
cur.execute("""
CREATE TABLE IF NOT EXISTS tarefas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    titulo TEXT NOT NULL,
    descricao TEXT,
    concluida INTEGER NOT NULL DEFAULT 0,
    usuario_id INTEGER,
    FOREIGN KEY (usuario_id) REFERENCES usuarios(id)
);
""")

con.commit()


# 3. CRUD – Operações Essenciais

A seguir, vamos implementar as funções de **Create**, **Read**, **Update** e **Delete** utilizando Python e SQLite.

---


📌 CREATE — Inserindo dados

In [ ]:
def inserir_usuario(nome, email):
    with sqlite3.connect("meubanco.db") as con:
        cur = con.cursor()
        cur.execute("INSERT INTO usuarios (nome, email) VALUES (?, ?)", (nome, email))
        con.commit()

def inserir_tarefa(titulo, descricao, usuario_id=None):
    with sqlite3.connect("meubanco.db") as con:
        cur = con.cursor()
        cur.execute("""
            INSERT INTO tarefas (titulo, descricao, concluida, usuario_id)
            VALUES (?, ?, 0, ?)
        """, (titulo, descricao, usuario_id))
        con.commit()


📌 READ — Consultando dados

In [ ]:
def listar_usuarios():
    with sqlite3.connect("meubanco.db") as con:
        cur = con.cursor()
        cur.execute("SELECT id, nome, email FROM usuarios")
        return cur.fetchall()

def listar_tarefas(usuario_id=None, apenas_concluidas=False):
    with sqlite3.connect("meubanco.db") as con:
        cur = con.cursor()

        sql = "SELECT id, titulo, descricao, concluida, usuario_id FROM tarefas"
        params = []

        if usuario_id is not None:
            sql += " WHERE usuario_id = ?"
            params.append(usuario_id)

        if apenas_concluidas:
            sql += " AND concluida = 1" if usuario_id is not None else " WHERE concluida = 1"

        cur.execute(sql, params)
        return cur.fetchall()


📌 UPDATE — Atualizando dados

In [ ]:
def marcar_concluida(id_tarefa):
    with sqlite3.connect("meubanco.db") as con:
        cur = con.cursor()
        cur.execute("UPDATE tarefas SET concluida = 1 WHERE id = ?", (id_tarefa,))
        con.commit()

def atualizar_usuario(id, novo_nome, novo_email):
    with sqlite3.connect("meubanco.db") as con:
        cur = con.cursor()
        cur.execute("""
            UPDATE usuarios SET nome = ?, email = ?
            WHERE id = ?
        """, (novo_nome, novo_email, id))
        con.commit()


📌 DELETE — Removendo registros

In [ ]:
def deletar_tarefa(id_tarefa):
    with sqlite3.connect("meubanco.db") as con:
        cur = con.cursor()
        cur.execute("DELETE FROM tarefas WHERE id = ?", (id_tarefa,))
        con.commit()

def deletar_usuario(id_usuario):
    with sqlite3.connect("meubanco.db") as con:
        cur = con.cursor()
        cur.execute("DELETE FROM tarefas WHERE usuario_id = ?", (id_usuario,))
        cur.execute("DELETE FROM usuarios WHERE id = ?", (id_usuario,))
        con.commit()


📌 Demonstração Completa

In [ ]:
# 3. Conectando ao SQLite e criando o banco
import sqlite3

BANCO = "db1.db"   # Usar o mesmo banco em todas as operações

# Cria ou conecta ao banco
con = sqlite3.connect(BANCO)
cur = con.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS usuarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS tarefas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    titulo TEXT NOT NULL,
    descricao TEXT,
    concluida INTEGER NOT NULL DEFAULT 0,
    usuario_id INTEGER,
    FOREIGN KEY (usuario_id) REFERENCES usuarios(id)
);
""")

con.commit()
con.close()


# ---------------------------------------------------------
# CREATE
# ---------------------------------------------------------
def inserir_usuario(nome, email):
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("INSERT INTO usuarios (nome, email) VALUES (?, ?)", (nome, email))
        con.commit()


def inserir_tarefa(titulo, descricao, usuario_id=None):
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("""
            INSERT INTO tarefas (titulo, descricao, concluida, usuario_id)
            VALUES (?, ?, 0, ?)
        """, (titulo, descricao, usuario_id))
        con.commit()


# ---------------------------------------------------------
# READ
# ---------------------------------------------------------
def listar_usuarios():
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("SELECT id, nome, email FROM usuarios")
        return cur.fetchall()


def listar_tarefas(usuario_id=None, apenas_concluidas=False):
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()

        sql = "SELECT id, titulo, descricao, concluida, usuario_id FROM tarefas"
        params = []

        if usuario_id is not None:
            sql += " WHERE usuario_id = ?"
            params.append(usuario_id)

        if apenas_concluidas:
            sql += " AND concluida = 1" if usuario_id is not None else " WHERE concluida = 1"

        cur.execute(sql, params)
        return cur.fetchall()


# ---------------------------------------------------------
# UPDATE
# ---------------------------------------------------------
def marcar_concluida(id_tarefa):
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("UPDATE tarefas SET concluida = 1 WHERE id = ?", (id_tarefa,))
        con.commit()


def atualizar_usuario(id, novo_nome, novo_email):
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("""
            UPDATE usuarios SET nome = ?, email = ?
            WHERE id = ?
        """, (novo_nome, novo_email, id))
        con.commit()


# ---------------------------------------------------------
# DELETE
# ---------------------------------------------------------
def deletar_tarefa(id_tarefa):
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("DELETE FROM tarefas WHERE id = ?", (id_tarefa,))
        con.commit()


def deletar_usuario(id_usuario):
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("DELETE FROM tarefas WHERE usuario_id = ?", (id_usuario,))
        cur.execute("DELETE FROM usuarios WHERE id = ?", (id_usuario,))
        con.commit()


# ---------------------------------------------------------
# TESTE PRÁTICO (Demonstração)
# ---------------------------------------------------------

print("=== INSERINDO USUÁRIOS ===")
inserir_usuario("Ana", "ana@example.com")
inserir_usuario("Bruno", "bruno@example.com")

print("\n=== INSERINDO TAREFAS ===")
inserir_tarefa("Estudar SQLite", "Revisar comandos básicos", 1)
inserir_tarefa("Enviar relatório", "Finalizar documento do projeto", 2)
inserir_tarefa("Fazer compras", "Ir ao mercado para compras da semana", 1)

print("\n=== LISTA DE USUÁRIOS ===")
for u in listar_usuarios():
    print(f"ID: {u[0]} | Nome: {u[1]} | Email: {u[2]}")

print("\n=== LISTA DE TAREFAS ===")
for t in listar_tarefas():
    print(f"ID: {t[0]} | Título: {t[1]} | Concluída: {t[3]} | Usuário ID: {t[4]}")

print("\n=== MARCANDO TAREFA 1 COMO CONCLUÍDA ===")
marcar_concluida(1)

print("\n=== DELETANDO TAREFA 2 ===")
deletar_tarefa(2)

print("\n=== TAREFAS APÓS ALTERAÇÕES ===")
for t in listar_tarefas():
    print(f"ID: {t[0]} | Título: {t[1]} | Concluída: {t[3]} | Usuário ID: {t[4]}")

print("\n=== ATUALIZANDO USUÁRIO 2 ===")
atualizar_usuario(2, "Bruno Silva", "bruno.silva@example.com")

print("\n=== USUÁRIOS ATUALIZADOS ===")
for u in listar_usuarios():
    print(f"ID: {u[0]} | Nome: {u[1]} | Email: {u[2]}")


# 4. Sistema de Gerenciamento de Cadastro de Usuário

In [ ]:
import sqlite3

BANCO = "sistema.db"

# ---------------------------------------------------------
# Cria ou conecta ao banco e cria a tabela se não existir
# ---------------------------------------------------------
def criar_tabela():
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS usuarios (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nome TEXT NOT NULL,
                email TEXT UNIQUE NOT NULL
            )
        """)
        con.commit()

# ---------------------------------------------------------
# Funções CRUD
# ---------------------------------------------------------
def inserir_usuario(nome, email):
    try:
        with sqlite3.connect(BANCO) as con:
            cur = con.cursor()
            cur.execute("INSERT INTO usuarios (nome, email) VALUES (?, ?)", (nome, email))
            con.commit()
            print(f"✔ Usuário inserido: {nome} - {email}")
    except sqlite3.IntegrityError:
        print(f"✖ Erro: o e-mail '{email}' já está cadastrado.")

def listar_usuarios():
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("SELECT id, nome, email FROM usuarios")
        return cur.fetchall()

def atualizar_usuario(id, novo_nome, novo_email):
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("UPDATE usuarios SET nome = ?, email = ? WHERE id = ?", (novo_nome, novo_email, id))
        con.commit()
        if cur.rowcount:
            print(f"✔ Usuário {id} atualizado.")
        else:
            print(f"✖ Não encontrou usuário com id {id}.")

def deletar_usuario(id):
    with sqlite3.connect(BANCO) as con:
        cur = con.cursor()
        cur.execute("DELETE FROM usuarios WHERE id = ?", (id,))
        con.commit()
        if cur.rowcount:
            print(f"✔ Usuário {id} excluído.")
        else:
            print(f"✖ Não encontrou usuário com id {id}.")

# ---------------------------------------------------------
# Utilitário para printar a lista de usuários
# ---------------------------------------------------------
def mostrar_usuarios():
    usuarios = listar_usuarios()
    if not usuarios:
        print("Nenhum usuário cadastrado.")
        return
    print("\n--- Usuários cadastrados ---")
    for u in usuarios:
        print(f"ID: {u[0]} | Nome: {u[1]} | Email: {u[2]}")
    print("----------------------------\n")

# ---------------------------------------------------------
# Execução de teste (somente quando o arquivo for executado diretamente)
# ---------------------------------------------------------
if __name__ == "__main__":
    criar_tabela()

    # Inserindo dados
    print("=== INSERINDO USUÁRIOS ===")
    inserir_usuario("Ana", "ana@email.com")
    inserir_usuario("Carlos", "carlos@email.com")

    # Listando usuários
    mostrar_usuarios()

    # Atualizando usuário
    print("=== ATUALIZANDO USUÁRIO 1 ===")
    atualizar_usuario(1, "Ana Souza", "ana.souza@email.com")
    mostrar_usuarios()

    # Deletando usuário
    print("=== DELETANDO USUÁRIO 2 ===")
    deletar_usuario(2)
    mostrar_usuarios()


Banco e tabela criados com sucesso!


# **5 - Sistema de Gerenciamento de Cadastro de Usuário com MENU INTERATIVO**

In [ ]:
import sqlite3

# =============================================================
# 1. Função para conectar ao banco
# =============================================================
def conectar():
    return sqlite3.connect("meubanco.db")


# =============================================================
# 2. Criando a tabela (caso não exista)
# =============================================================
con = conectar()
cur = con.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS usuarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL
)
""")

con.commit()
con.close()

print("Banco e tabela criados com sucesso!")
print("-" * 50)


# =============================================================
# 3. Funções CRUD
# =============================================================

def inserir_usuario():
    nome = input("Digite o nome do usuário: ")
    email = input("Digite o e-mail do usuário: ")

    try:
        con = conectar()
        cur = con.cursor()
        cur.execute("INSERT INTO usuarios (nome, email) VALUES (?, ?)", (nome, email))
        con.commit()
        print("Usuário cadastrado com sucesso!")
    except sqlite3.IntegrityError:
        print("Erro: este e-mail já está cadastrado.")
    finally:
        con.close()


def listar_usuarios():
    con = conectar()
    cur = con.cursor()
    cur.execute("SELECT * FROM usuarios")
    usuarios = cur.fetchall()
    con.close()

    if not usuarios:
        print("Nenhum usuário cadastrado.")
    else:
        print("\n--- LISTA DE USUÁRIOS ---")
        for u in usuarios:
            print(f"ID: {u[0]} | Nome: {u[1]} | Email: {u[2]}")
        print("--------------------------")


def buscar_por_nome():
    nome = input("Digite o nome que deseja buscar: ")

    con = conectar()
    cur = con.cursor()
    cur.execute("SELECT * FROM usuarios WHERE nome LIKE ?", (f"%{nome}%",))
    usuarios = cur.fetchall()
    con.close()

    if not usuarios:
        print("Nenhum usuário encontrado.")
    else:
        print("\nUsuários encontrados:")
        for u in usuarios:
            print(f"ID: {u[0]} | Nome: {u[1]} | Email: {u[2]}")


def atualizar_usuario():
    nome = input("Digite o nome do usuário que deseja atualizar: ")

    con = conectar()
    cur = con.cursor()
    cur.execute("SELECT * FROM usuarios WHERE nome LIKE ?", (f"%{nome}%",))
    usuario = cur.fetchone()

    if not usuario:
        print("Usuário não encontrado.")
        con.close()
        return

    print(f"Usuário encontrado: {usuario[1]} - {usuario[2]}")

    novo_nome = input("Digite o novo nome: ")
    novo_email = input("Digite o novo e-mail: ")

    try:
        cur.execute("UPDATE usuarios SET nome=?, email=? WHERE id=?", (novo_nome, novo_email, usuario[0]))
        con.commit()
        print("Usuário atualizado com sucesso!")
    except sqlite3.IntegrityError:
        print("Erro: este e-mail já está cadastrado em outro usuário.")
    finally:
        con.close()


def deletar_usuario():
    nome = input("Digite o nome do usuário que deseja excluir: ")

    con = conectar()
    cur = con.cursor()
    cur.execute("SELECT * FROM usuarios WHERE nome LIKE ?", (f"%{nome}%",))
    usuario = cur.fetchone()

    if not usuario:
        print("Usuário não encontrado.")
        con.close()
        return

    print(f"Usuário encontrado: {usuario[1]} - {usuario[2]}")
    confirmar = input("Tem certeza que deseja excluir? (s/n): ").lower()

    if confirmar == "s":
        cur.execute("DELETE FROM usuarios WHERE id=?", (usuario[0],))
        con.commit()
        print("Usuário excluído com sucesso!")
    else:
        print("Operação cancelada.")

    con.close()


# =============================================================
# 4. MENU INTERATIVO
# =============================================================
def menu():
    while True:
        print("""
=========== MENU ===========

1 - Cadastrar usuário
2 - Listar usuários
3 - Buscar usuário por nome
4 - Atualizar usuário pelo nome
5 - Excluir usuário pelo nome
0 - Sair

============================
""")

        opc = input("Escolha uma opção: ")

        if opc == "1":
            inserir_usuario()
        elif opc == "2":
            listar_usuarios()
        elif opc == "3":
            buscar_por_nome()
        elif opc == "4":
            atualizar_usuario()
        elif opc == "5":
            deletar_usuario()
        elif opc == "0":
            print("Saindo... Até mais!")
            break
        else:
            print("Opção inválida! Tente novamente.")


# Executa o menu
menu()



## 6. Exercícios Propostos

1. Criar uma tabela **produtos** com os campos: `id`, `nome`, `preco`.
2. Inserir pelo menos **3 produtos** na tabela.
3. Criar uma função para **buscar produto pelo nome**.
4. Criar uma função para **aumentar o preço de todos os produtos em 10%**.
5. Criar uma função para **excluir todos os produtos com preço abaixo de R$ 5,00**.
6. Criar um menu interativo (usando `while`) para que o usuário escolha as opções do CRUD no console.
